# Framework

In [ ]:
import pandas as pd
from textblob import TextBlob
import emoji
import re

import torch
from torch.utils.data import DataLoader, TensorDataset
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification, AdamW
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import pandas as pd
from textblob import TextBlob
import re
import emoji
import matplotlib.pyplot as plt
from sklearn.metrics import  accuracy_score, precision_score, f1_score, recall_score
from tqdm import tqdm
from sklearn.model_selection import ParameterGrid
from torch.nn.utils.rnn import pad_sequence

# Data Exploration

In [ ]:
# Load dataset
hatespeech_train = pd.read_csv('hatespeech_train.csv')

# Display the first few rows of the dataset
print(hatespeech_train.head())

In [ ]:
# Check the exact name of your label column
label_column_name = 'label'  # Replace with the actual name

# Calculate EDA metrics on Label
label_stats = hatespeech_train[label_column_name].describe()

# Calculate additional metrics
label_stats['mean'] = hatespeech_train[label_column_name].mean()
label_stats['median'] = hatespeech_train[label_column_name].median()
label_stats['std_dev'] = hatespeech_train[label_column_name].std()
label_stats['skewness'] = hatespeech_train[label_column_name].skew()

# Display the calculated metrics
print(label_stats)

In [ ]:
# Plot label
hatespeech_train[label_column_name].value_counts().plot(kind='bar', title='Label Distribution')
plt.xlabel('Label')
plt.ylabel('Count')
plt.xticks([0, 1], ['Not Hate Speech (0)', 'Hate Speech (1)'], rotation=0)
plt.show()

# Data Pre-Processing

In [ ]:
#Dataset Definition
twits_25k_path = '25k_dataset.csv'
twits_32k_path = 'hatespeech_train.csv'
twits_150_path = 

twits_25k = pd.read_csv(twits_25k_path)
twits_32k = pd.read_csv(twits_32k_path)

In [ ]:
def preprocess_dataset(file_path):
    # Load dataset
    dataset = pd.read_csv(file_path)
    # Change text to lowercase
    dataset['tweet'] = dataset['tweet'].str.lower()
    # Eliminate URLs
    dataset['tweet'] = dataset['tweet'].replace(r'http\S+', '', regex=True).replace(r'www\S+', '', regex=True)
    # Use TextBlob for spelling correction
    dataset['tweet'] = dataset['tweet'].apply(lambda x: str(TextBlob(x).correct()))
    # Convert emojis to text and keep sentiment
    dataset['tweet'] = dataset['tweet'].apply(lambda x: emoji.demojize(x))
    # Keep hashtags
    dataset['tweet'] = dataset['tweet'].apply(lambda x: re.sub(r'#', '', x))
    # Eliminate @mentions
    dataset['tweet'] = dataset['tweet'].replace(r'@\S+', '', regex=True)
    return dataset

# Uso de la función con tu ruta de archivo
hatespeech_train_preprocessed = preprocess_dataset('hatespeech_train.csv')

# Muestra las primeras filas del DataFrame resultante
print(hatespeech_train_preprocessed.head())


In [ ]:
# Load dataset
hatespeech_train = pd.read_csv('hatespeech_train.csv')

# Display the first few rows of the dataset
print(hatespeech_train.head())

In [ ]:
# Change text to lowercase
hatespeech_train['tweet'] = hatespeech_train['tweet'].str.lower()

# Eliminate URLs
hatespeech_train['tweet'] = hatespeech_train['tweet'].replace(r'http\S+', '', regex=True).replace(r'www\S+', '', regex=True)


In [ ]:
# Use TextBlob for spelling correction
hatespeech_train['tweet'] = hatespeech_train['tweet'].apply(lambda x: str(TextBlob(x).correct()))

In [ ]:
# Convert emojis to text and keep sentiment
hatespeech_train['tweet'] = hatespeech_train['tweet'].apply(lambda x: emoji.demojize(x))

In [ ]:
# Keep hashtags
hatespeech_train['tweet'] = hatespeech_train['tweet'].apply(lambda x: re.sub(r'#', '', x))

In [ ]:
# Eliminate @mentions
hatespeech_train['tweet'] = hatespeech_train['tweet'].replace(r'@\S+', '', regex=True)

In [ ]:
hatespeech_train.head()

In [ ]:
# Save the prepared DataFrame to a CSV file
hatespeech_train.to_csv('/Users/lauraguinandrodriguez/Desktop/Hate Speech/prepared_hatespeech_train.csv', index=False)

print(f"Prepared data saved to {'/Users/lauraguinandrodriguez/Desktop/Hate Speech/prepared_hatespeech_train.csv'}")

Train and Test Split

In [ ]:
# Split the dataset into train (80%) and test (20%)
train_df, test_df = train_test_split(hatespeech_train, test_size=0.2, random_state=42)

# Display the first few rows of the preprocessed dataset
print(train_df.head())
print(test_df.head())

In [ ]:
# Save train and test datasets to CSV files
train_df.to_csv('train_dataset.csv', index=False)
test_df.to_csv('test_dataset.csv', index=False)

# DistilBERT Pre-trained Model

## Load Train and Test Datasets

Issues with tokenization made in the data preparation so they were implemented below in the cell that runs the model. 
To reduce time, droped Tokens column when calling the current saved csv and re-ran the tokenization and label encoding below. 

In [ ]:
# Load dataset
hatespeech_train = pd.read_csv('/Users/lauraguinandrodriguez/Desktop/Hate Speech/train_dataset.csv')

# Drop the 'Tokens' column
hatespeech_train = hatespeech_train.drop(columns=['Tokens'])

# Display the first few rows of the modified dataset
print(hatespeech_train.head())

In [ ]:
# Load dataset
hatespeech_test = pd.read_csv('/Users/lauraguinandrodriguez/Desktop/Hate Speech/test_dataset.csv')

# Drop the 'Tokens' column
hatespeech_test = hatespeech_test.drop(columns=['Tokens'])

# Display the first few rows of the modified dataset
print(hatespeech_test.head())

## Tokenization, Label encoding, Hyperparameter Tunning, GridSearch, Evaluation

In [ ]:
# Tokenization
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
hatespeech_train['Tokens'] = hatespeech_train['tweet'].apply(lambda x: tokenizer.encode(x, add_special_tokens=True))
hatespeech_test['Tokens'] = hatespeech_test['tweet'].apply(lambda x: tokenizer.encode(x, add_special_tokens=True))

# Label Encoding
label_encoder = LabelEncoder()
hatespeech_train['label'] = label_encoder.fit_transform(hatespeech_train['label'])
hatespeech_test['label'] = label_encoder.transform(hatespeech_test['label'])

# Padding sequences
token_tensors_train = [torch.tensor(token_ids) for token_ids in hatespeech_train['Tokens']]
token_tensors_test = [torch.tensor(token_ids) for token_ids in hatespeech_test['Tokens']]

padded_token_tensors_train = pad_sequence(token_tensors_train, batch_first=True, padding_value=0)
padded_token_tensors_test = pad_sequence(token_tensors_test, batch_first=True, padding_value=0)

# Create TensorDatasets
dataset_train = TensorDataset(padded_token_tensors_train, torch.tensor(hatespeech_train['label'].tolist()))
dataset_test = TensorDataset(padded_token_tensors_test, torch.tensor(hatespeech_test['label'].tolist()))

# Hyperparameter Grid Search
param_grid = {
    'learning_rate': [1e-5, 3e-5],
    'epochs': [3, 7],
    'batch_size': [16, 32]
}

best_accuracy = 0
best_hyperparameters = None
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# DataLoader instances for training and validation
train_dataloader = DataLoader(dataset_train, batch_size=batch_size, shuffle=True)
validation_dataloader = DataLoader(dataset_test, batch_size=batch_size)

# Loop through hyperparameter combinations
for params in tqdm(list(ParameterGrid(param_grid))):
    model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased').to(device)
    optimizer = AdamW(model.parameters(), lr=params['learning_rate'])

    # Training loop
    for epoch in range(params['epochs']):
        model.train()
        total_loss = 0

        # Iterate over batches
        for batch in tqdm(train_dataloader, desc=f"Epoch {epoch + 1}/{params['epochs']}"):
            inputs, labels = batch
            inputs, labels = inputs.to(device), labels.to(device)

            # Forward pass
            outputs = model(inputs, labels=labels)  # Specify labels during training
            loss = outputs.loss

            # Backward pass and optimization
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        # Calculate average training loss for the epoch
        average_loss = total_loss / len(train_dataloader)
        print(f"Epoch {epoch + 1}/{params['epochs']}, Average Training Loss: {average_loss}")

    # Evaluation (Move these lines outside the training loop)
    model.eval()
    with torch.no_grad():
        total_metrics = {'accuracy': 0, 'precision': 0, 'recall': 0, 'f1': 0}
        total_samples = 0

        for val_batch in tqdm(validation_dataloader, desc=f"Evaluation"):
            val_inputs, val_labels = val_batch
            val_inputs, val_labels = val_inputs.to(device), val_labels.to(device)

            # Forward pass
            val_outputs = model(val_inputs)
            val_predictions = val_outputs.logits.argmax(dim=-1)

            # Update metrics
            total_metrics['accuracy'] += accuracy_score(val_labels.cpu(), val_predictions.cpu()) * len(val_labels)
            total_metrics['precision'] += precision_score(val_labels.cpu(), val_predictions.cpu(), average='weighted', zero_division=1.0) * len(val_labels)
            total_metrics['recall'] += recall_score(val_labels.cpu(), val_predictions.cpu(), average='weighted', zero_division=1.0) * len(val_labels)
            total_metrics['f1'] += f1_score(val_labels.cpu(), val_predictions.cpu(), average='weighted', zero_division=1.0) * len(val_labels)

            total_samples += len(val_labels)

        # Calculate average metrics for the epoch
        average_metrics = {key: total_metrics[key] / total_samples for key in total_metrics}
        print(f"Epoch {epoch + 1}/{params['epochs']}, Evaluation Metrics: {average_metrics}")

    # Check if current hyperparameters result in better accuracy
    if average_metrics['accuracy'] > best_accuracy:
        best_accuracy = average_metrics['accuracy']
        best_hyperparameters = params

# Print the best hyperparameters and corresponding accuracy
print("Best Hyperparameters:", best_hyperparameters)
print("Best Accuracy:", best_accuracy)
